# Electron g-2 – 24-Cell Generalization Test

Testing the same interference model on the 24-cell lattice (24 vertices, degree ~8).
Goal: check if strong δμ_e suppression persists on a different 4D regular polytope.

Parameters: starting guess tuned for smaller lattice size.
Weighting: 1/(step+1)^{1.8}
Paths per run: 1 billion × 10 repeats (quick test; increase later)

In [ ]:
import cupy as cp
import numpy as np
from tqdm import tqdm
import time
import pandas as pd
import winsound

# ====================== SETTINGS ======================
N_paths_total = 1_000_000_000   # 1 billion per run – increase to 5B+ later
BATCH_SIZE    = 25_000_000      # safe starting point
N_REPEATS     = 10              # quick test

C = 9.6e-7
alpha = 1 / 137.035999084

configs = [
    {'name': '24-Cell Test: 22/32/7.2 weighted', 'walk_length': 22, 'phase_scale': 32.0, 'fbs_power': 7.2},
]

n_batches = (N_paths_total + BATCH_SIZE - 1) // BATCH_SIZE
print(f"Total paths per run: {N_paths_total:,} | Repeats: {N_REPEATS}")

In [ ]:
# ====================== 24-CELL VERTICES ======================
def generate_24cell_vertices():
    coords = []
    # All even permutations of (±1, ±1, 0, 0)
    for i in range(4):
        for j in range(i+1, 4):
            for si in [-1, 1]:
                for sj in [-1, 1]:
                    v = np.zeros(4)
                    v[i] = si
                    v[j] = sj
                    coords.append(v)
    verts = np.unique(np.array(coords), axis=0)  # 24 points
    verts = verts / np.linalg.norm(verts, axis=1)[:, None]  # unit norm
    return cp.asarray(verts, dtype=cp.float32)

vertices = generate_24cell_vertices()
N_VERT = vertices.shape[0]
print(f"Generated {N_VERT} vertices (expected 24)")

In [ ]:
# ====================== ADJACENCY ======================
dists = cp.linalg.norm(vertices[:, None] - vertices[None, :], axis=-1)
# 24 verts × 8 neighbors = 96 edges (directed 192 connections)
edge_threshold = cp.sort(dists.flatten())[192 + 1]  # after smallest 192 non-zero
adj_mask = (dists < edge_threshold + 1e-6) & (dists > 1e-6)
neighbors = [cp.argwhere(adj_mask[i])[:,0] for i in range(N_VERT)]
degrees = [len(n) for n in neighbors]
print(f"Average degree: {np.mean(degrees):.2f} (expected ~8)")
print(f"Min/Max degree: {min(degrees)} / {max(degrees)}")

In [ ]:
# ====================== RUN TEST ======================
all_results = []
global_start = time.time()

cfg = configs[0]
print(f"\n=== Running: {cfg['name']} ===")

for rep in range(1, N_REPEATS + 1):
    print(f"  Repeat {rep}/{N_REPEATS}")
    start = time.time()

    raw_amplitudes = cp.zeros(3, dtype=cp.complex64)

    for batch_idx in tqdm(range(n_batches), desc="Batches", leave=False):
        batch_size = min(BATCH_SIZE, N_paths_total - batch_idx * BATCH_SIZE)
        current = cp.random.randint(0, N_VERT, batch_size)
        phase = cp.zeros(batch_size, dtype=cp.complex64)
        fbs_grade = cp.ones(batch_size, dtype=cp.float32)

        for step in range(cfg['walk_length']):
            neigh_choices = cp.array([cp.random.choice(n, size=1)[0] for n in neighbors])[current]
            edge_len = cp.linalg.norm(vertices[current] - vertices[neigh_choices], axis=1)
            phase += 1j * cfg['phase_scale'] * edge_len * fbs_grade
            fbs_grade *= 1.0 / (edge_len ** cfg['fbs_power'] + 0.1)
            current = neigh_choices

        amp = cp.exp(phase) * fbs_grade

        # Path-length weighting
        weight = 1.0 / (step + 1) ** 1.8
        amp *= weight

        e_amp = amp * cp.exp(1j * 0.0)
        h_amp = amp * cp.exp(1j * np.pi / 2)
        q_amp = amp * cp.exp(1j * np.pi)

        raw_amplitudes[0] += cp.sum(e_amp)
        raw_amplitudes[1] += cp.sum(h_amp)
        raw_amplitudes[2] += cp.sum(q_amp)

        cp.cuda.Device().synchronize()

    interference = raw_amplitudes / N_paths_total
    abs_interf = cp.abs(interference)
    total_mag = cp.sum(abs_interf)
    mean_amp = float(cp.mean(cp.abs(amp)))

    S = alpha / (2 * np.pi) * mean_amp
    mixing_sum = float(abs_interf[2] / total_mag) + 0.7 * float(abs_interf[1] / total_mag)
    delta_mu_e = C * mixing_sum * S

    result = {
        'config': cfg['name'],
        'repeat': rep,
        'delta_mu_e': delta_mu_e,
        'mean_amp': mean_amp,
        'S': S,
        'raw_sum_mag': float(cp.abs(raw_amplitudes).sum()),
        'time_sec': time.time() - start
    }
    all_results.append(result)

    print(f"  δμ_e: {delta_mu_e:.3e} | mean_amp: {mean_amp:.3e}")

df = pd.DataFrame(all_results)
df.to_csv("24cell_test_results.csv", index=False)

print("\n=== SUMMARY ===")
print(f"Mean δμ_e:    {df['delta_mu_e'].mean():.3e}")
print(f"Median δμ_e:  {df['delta_mu_e'].median():.3e}")
print(f"Std δμ_e:     {df['delta_mu_e'].std():.3e}")
print(f"Best δμ_e:    {df['delta_mu_e'].min():.3e}")
print(f"Worst δμ_e:   {df['delta_mu_e'].max():.3e}")
print(f"Total time:   {(time.time() - global_start)/3600:.2f} hours")

# Beep
winsound.Beep(1200, 2000)
print("FINISHED!")

## Sorted Results

In [ ]:
df = pd.read_csv("24cell_test_results.csv")
print("Sorted by δμ_e (lowest first):")
print(df.sort_values('delta_mu_e').to_string(index=False))